# Chapter 23 — Compaction

**Companion to *Pi Agents*** · [`Pi Agents` chapter 23](https://programmer.ie/books/pi/23-chapter/)

| | |
|---|---|
| Chapter | `23-chapter.md` · weight 23 · `/books/pi/23-chapter/` |
| Notebook | `notebooks/pi/23-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

After a manual compaction, which history did the next request contain, and what
is still in the file?

The chapter's point: compaction **appends a summary entry** and **rebuilds the
context** for the next request from that summary plus the kept messages. The
session file is **never deleted** — navigating back to before the compaction
sends the original messages again.


## What this notebook establishes

- Compaction finds a **cut point** by walking backwards until `keepRecentTokens` is reached, then summarizes the replaced messages and appends a `compaction` entry with `firstKeptEntryId`.
- The **next request** is built from the summary plus messages from `firstKeptEntryId` onwards — the oldest messages are replaced by the summary.
- The session file **never loses entries** — navigating to before the compaction sends the original messages again.
- The **cut point is never a tool result** — tool calls and their results stay together (overshooting the budget if needed).
- Compaction is **iterative**: the second compaction receives the first's summary as `previousSummary`.
- An extension can **supply the summary** via `session_before_compact` (cut point stays Pi's, `fromHook: true`), or **cancel** a manual `/compact`.
- A **split user-message span** produces `isSplitTurn: true`, `messagesToSummarize` empty, and `turnPrefixMessages` for the early part — the shipped `custom-compaction.ts` joins both lists.

## What this notebook does **not** establish

- **No summary quality is evaluated** (`ch23-lim2`) — the extension's custom summary is a placeholder, and the test only checks that it arrived.
- **The split-turn counts in the prose** come from a `console.log`, not an assertion.
- **File operations tracking** (`readFiles`, `modifiedFiles`) is cumulative only for Pi-generated summaries; extension summaries with `fromHook: true` must manage their own `details`.
- **The `keepRecentTokens` value** in the driver is controlled by `NB_KEEP` (default 900) to make the test exercise compaction quickly.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — a real persisted session with the chapter's custom-summary extension, `keepRecentTokens` lowered so a short scripted conversation has something to summarise.
* **Evidence scope:** `in_process_runtime`. The compaction arithmetic, cut point rules, and extension hook are Pi's.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(23))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/23-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: before compaction

In [3]:
COMP = pinb.driver_source("ch23-replace.ts")
comp = pinb.driver(COMP, label="ch23-replace", timeout=300, env={"NB_KEEP": "900"})
print(pinb.md_table(
    ["metric", "before", "after"],
    [["entries", comp["before"]["entries"], comp["after"]["entries"]],
     ["lines in file", comp["before"]["lines"], comp["after"]["lines"]],
     ["messages", comp["before"]["messages"], comp["after"]["messages"]],
     ["compactions", comp["before"]["compactions"], comp["after"]["compactions"]],
     ["question A in file", "yes" if comp["before"]["questionAInFile"] else "no", "yes" if comp["after"]["questionAInFile"] else "no"]],
))

| metric | before | after |
|---|---|---|
| entries | 11 | 12 |
| lines in file | 12 | 13 |
| messages | 9 | 9 |
| compactions | 0 | 1 |
| question A in file | yes | yes |


In [4]:
pinb.show_checks([
    pinb.check("compaction adds one entry", comp["after"]["entries"] == comp["before"]["entries"] + 1, "entry count +1"),
    pinb.check("file grows by one line", comp["after"]["lines"] == comp["before"]["lines"] + 1, "file line +1"),
    pinb.check("question A still in file after compaction", comp["after"]["questionAInFile"], "not deleted"),
])

== Assertions ==
  assertion  observed
----  -----------------------------------------  --------------
PASS  compaction adds one entry  entry count +1
PASS  file grows by one line  file line +1
PASS  question A still in file after compaction  not deleted

3/3 assertions held.


## The compaction entry

In [5]:
c = comp["compaction"]
print(pinb.md_table(
    ["field", "value"],
    [["fromHook", "yes" if c["fromHook"] else "no"],
     ["summary is extension's", "yes" if c["summaryIsTheExtensions"] else "no"],
     ["has firstKeptEntryId", "yes" if c["hasFirstKeptEntryId"] else "no"],
     ["firstKept role", c["firstKeptRole"] or "null"],
     ["firstKept is toolResult", "yes" if c["firstKeptIsToolResult"] else "no"],
     ["tokensBefore positive", "yes" if c["tokensBeforeIsPositive"] else "no"]],
))

| field | value |
|---|---|
| fromHook | yes |
| summary is extension's | yes |
| has firstKeptEntryId | yes |
| firstKept role | assistant |
| firstKept is toolResult | no |
| tokensBefore positive | yes |


In [6]:
pinb.show_checks([
    pinb.check("extension supplied the summary", c["fromHook"], "fromHook: true"),
    pinb.check("summary text is from the extension", c["summaryIsTheExtensions"], "custom summary used"),
    pinb.check("firstKeptEntryId is set", c["hasFirstKeptEntryId"], "cut point recorded"),
    pinb.check("firstKept is a user message (legal cut point)", not c["firstKeptIsToolResult"], "not a tool result"),
    pinb.check("tokensBefore was recalculated", c["tokensBeforeIsPositive"], "tokensBefore > 0"),
])

== Assertions ==
  assertion  observed
----  ---------------------------------------------  -------------------
PASS  extension supplied the summary  fromHook: true
PASS  summary text is from the extension  custom summary used
PASS  firstKeptEntryId is set  cut point recorded
PASS  firstKept is a user message (legal cut point)  not a tool result
PASS  tokensBefore was recalculated  tokensBefore > 0

5/5 assertions held.


## What the next request contained

In [7]:
s = comp["sent"]
print(pinb.md_table(
    ["metric", "value"],
    [["roles in request", " -> ".join(s["roles"])],
     ["user texts (truncated)", ", ".join(s["userTexts"])],
     ["question A sent", "yes" if s["questionASent"] else "no"],
     ["question C sent", "yes" if s["questionCSent"] else "no"],
     ["summary arrived as user message", "yes" if s["summaryArrivedAsUserMessage"] else "no"],
     ["tool calls in request", s["toolCalls"]],
     ["tool results in request", s["toolResults"]],
     ["system still declares bash", "yes" if s["systemStillDeclaresBash"] else "no"]],
))

| metric | value |
|---|---|
| roles in request | user -> assistant -> user -> assistant -> toolResult -> assistant -> user -> assistant -> user |
| user texts (truncated) | question B, question C, next question |
| question A sent | no |
| question C sent | yes |
| summary arrived as user message | yes |
| tool calls in request | 1 |
| tool results in request | 1 |
| system still declares bash | yes |


In [8]:
pinb.show_checks([
    pinb.check("oldest question (A) is NOT sent — replaced by summary", not s["questionASent"], "A replaced"),
    pinb.check("newest question (C) IS sent — in kept range", s["questionCSent"], "C kept"),
    pinb.check("summary arrives as user-role message", s["summaryArrivedAsUserMessage"], "user message"),
    pinb.check("tool calls = tool results (no half calls)", s["toolCalls"] == s["toolResults"], "balanced"),
    pinb.check("system prompt still declares bash (not shrunk)", s["systemStillDeclaresBash"], "bash declared"),
])

== Assertions ==
  assertion  observed
----  -----------------------------------------------------  -------------
PASS  oldest question (A) is NOT sent — replaced by summary  A replaced
PASS  newest question (C) IS sent — in kept range  C kept
PASS  summary arrives as user-role message  user message
PASS  tool calls = tool results (no half calls)  balanced
PASS  system prompt still declares bash (not shrunk)  bash declared

5/5 assertions held.


## Navigating back to before compaction

In [9]:
nb = comp["navigatedBack"]
print(pinb.md_table(
    ["what was sent", "value"],
    [["question A sent again", "yes" if nb["questionASentAgain"] else "no"],
     ["summary sent", "yes" if nb["summarySent"] else "no"]],
))

| what was sent | value |
|---|---|
| question A sent again | yes |
| summary sent | no |


In [10]:
pinb.show_checks([
    pinb.check("navigating back sends original question A", nb["questionASentAgain"], "original restored"),
    pinb.check("navigating back does NOT send the summary", not nb["summarySent"], "summary not sent"),
    pinb.check("file still has question A entry", comp["fileStillHasQuestionA"], "file unchanged"),
])

== Assertions ==
  assertion  observed
----  -----------------------------------------  -----------------
PASS  navigating back sends original question A  original restored
PASS  navigating back does NOT send the summary  summary not sent
PASS  file still has question A entry  file unchanged

3/3 assertions held.


## The chapter's own tests

All tests from both test files.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [11]:
PATTERNS = ['ch23-compaction/compaction.test.ts', 'ch23-compaction/state.test.ts']
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch23-compaction/compaction.test.ts, ch23-compaction/state.test.ts', show="compaction")

$ node --test ch23-compaction/compaction.test.ts, ch23-compaction/state.test.ts
  PASS  12 passed, 0 failed, 0 skipped  in 2206 ms

  tests matching 'compaction':
    ok   compaction replaces what the model sees and deletes nothing from the file
    ok   an extension can cancel a manual compaction
    ok   compaction is iterative: the second summary is given the first as previousSummary
    ok   compaction does not shrink the system prompt or the tool declarations
    ok   compaction replaces what the model reads and nothing in the file: navigating to before the compaction sends the originals again


## Your turn: predict, then run

**Predict first.** What happens if you set `NB_KEEP=20000` (the default) instead of
900? Does compaction still fire for the same three-turn history?

**Then change one input.** The extension's custom summary includes only the first
500 characters of the serialized conversation. What happens if you increase that
limit or use the full serialization?

**Predict the boundary.** The split-turn case: one user message followed by six
tool-calling turns. The driver doesn't exercise this — what would the extension
need to do differently to handle `turnPrefixMessages`?

In [12]:
print("Predictions:")
print("  1. NB_KEEP=20000: no compaction — the short history fits in the budget.")
print("  2. Larger summary limit: more detail in summary, but output token cap still binds.")
print("  3. Split turn: extension must join turnPrefixMessages + messagesToSummarize, not just the latter.")
print()
print("Observed above:")
print(f"  keepRecentTokens: {comp["keepRecentTokens"]}")
print(f"  compaction entry: fromHook={comp["compaction"]["fromHook"]}, firstKept={comp["compaction"]["firstKeptRole"]}")
print(f"  next request: A sent={comp["sent"]["questionASent"]}, C sent={comp["sent"]["questionCSent"]}")
print(f"  navigate back: A again={comp["navigatedBack"]["questionASentAgain"]}, summary={comp["navigatedBack"]["summarySent"]}")
print()
assert comp["compaction"]["fromHook"]
assert not comp["sent"]["questionASent"]
assert comp["sent"]["questionCSent"]
assert comp["navigatedBack"]["questionASentAgain"]
assert not comp["navigatedBack"]["summarySent"]
print("held: compaction replaces old messages with summary; file is unchanged; navigate back restores originals")

Predictions:
  1. NB_KEEP=20000: no compaction — the short history fits in the budget.
  2. Larger summary limit: more detail in summary, but output token cap still binds.
  3. Split turn: extension must join turnPrefixMessages + messagesToSummarize, not just the latter.

Observed above:
  keepRecentTokens: 900
  compaction entry: fromHook=True, firstKept=assistant
  next request: A sent=False, C sent=True
  navigate back: A again=True, summary=False

held: compaction replaces old messages with summary; file is unchanged; navigate back restores originals


## Interpretation

Compaction is three things, and the chapter shows the boundaries:

| Mechanism | What it does | The boundary |
|---|---|---|
| Cut point search | Walk back until `keepRecentTokens` | **Never cuts at tool results** — overshoots budget to keep call+result together |
| Summary generation | LLM call with structured format | **Iterative** — `previousSummary` passed to next compaction |
| Context rebuild | Summary + kept messages | **Original entries stay in file** — navigate back restores them |

The extension hook (`session_before_compact`):
* **Supply summary**: return `{ compaction: { summary, firstKeptEntryId, tokensBefore } }` — Pi keeps the cut point, marks `fromHook: true`
* **Cancel**: return `{ cancel: true }` — only for `reason === "manual"`

The split-turn case:
* One long user-message span exceeding `keepRecentTokens` → `isSplitTurn: true`
* `messagesToSummarize` empty, early part in `turnPrefixMessages`
* Pi generates two summaries and merges them
* **Extension must handle both lists** — the shipped `custom-compaction.ts` joins them

Practical rules:
1. **Compaction does not delete** — navigate back and the originals are there.
2. **Put durable facts in context files (ch 28)** — history is summarised, context files travel with every request.
3. **If you supply a summary, include file lists** — `details.readFiles`/`modifiedFiles` are not auto-recovered from `fromHook: true` summaries.
4. **`keepRecentTokens` controls granularity** — lower = more frequent, smaller summaries; higher = fewer, larger summaries.

## Sources, execution mode and limitations

**Execution mode:** **Run** — a real persisted session with the chapter's custom-summary extension, `keepRecentTokens` lowered so a short scripted conversation has something to summarise.

**Evidence scope:** `in_process_runtime`. The compaction arithmetic, cut point rules, and extension hook are Pi's.

### What was read or run

- **Ran** `drivers/ch23-replace.ts`: real session with custom summary, compaction, navigation back.
- **Ran** `ch23-compaction/compaction.test.ts` and `ch23-compaction/state.test.ts` (tests across multiple `keepRecentTokens` values).
- **Read** `examples/evidence.json`, chapter 23 rows (12 claims).

### Limitations

- **No summary quality is evaluated** (`ch23-lim2`) — the extension's custom summary is a placeholder.
- **The split-turn counts in the prose** come from a `console.log`, not an assertion.
- **File operations tracking** is cumulative only for Pi-generated summaries; extension summaries with `fromHook: true` must manage their own `details`.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
